# HEPTAPOD MCP Tutorial

This notebook walks through exposing HEPTAPOD's particle physics tools as **MCP (Model Context Protocol)** servers. MCP is an open protocol that lets AI assistants like Claude connect to external tools and data sources.

By the end of this tutorial you will be able to:
1. Explore HEPTAPOD's tool groups and inspect their schemas
2. Start an MCP server over **STDIO** (for Claude Code / Claude Desktop)
3. Start an MCP server over **HTTP** (for remote or multi-client access)
4. Register the servers with **Claude Code**
5. Verify the servers work by connecting as a client

---
## 1. Setup

Make sure the `mcp` Python package is installed (required for MCP server functionality):

```bash
pip install mcp
```

The `orchestral-ai` package should already be available if you have HEPTAPOD set up.

In [ ]:
import sys
from pathlib import Path

# Add repository root to path
REPO_ROOT = Path(".").resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

# Verify imports
import orchestral
print(f"Orchestral loaded from: {orchestral.__file__}")

try:
    import mcp
    print(f"MCP SDK available")
except ImportError:
    print("MCP SDK not found — install with: pip install mcp")

---
## 2. Exploring HEPTAPOD Tools

HEPTAPOD tools are organized into **groups**. Each group contains related tools that share a theme (particle data, literature search, analysis, etc.).

Let's import the tool registry and see what's available.

In [ ]:
from heptapod_tools import TOOL_GROUPS, get_tools, get_available_groups, LIGHTWEIGHT_GROUPS

print("All defined tool groups:")
for name in TOOL_GROUPS:
    marker = "  (lightweight)" if name in LIGHTWEIGHT_GROUPS else "  (requires external software)"
    print(f"  {name}{marker}")

print(f"\nGroups that can be loaded on this machine:")
for name in get_available_groups():
    print(f"  {name}")

### Loading a specific group

Let's load the PDG tools and inspect their schemas. This is exactly what an MCP client sees when it connects.

In [ ]:
import json

pdg_tools = get_tools("pdg")

for tool in pdg_tools:
    spec = tool.get_tool_spec()
    print(f"Tool: {spec.name}")
    print(f"Description: {spec.description[:120]}...")
    print(f"Input Schema:")
    print(json.dumps(spec.input_schema, indent=2))
    print()

### Loading multiple groups at once

In [ ]:
# Load a few lightweight groups
tools = get_tools("pdg", "units", "nda")

print(f"Loaded {len(tools)} tools from 3 groups:\n")
for tool in tools:
    print(f"  {tool.get_name():40s} {tool.__class__.__name__}")

### Loading all available tools

In [ ]:
all_tools = get_tools()  # No arguments = load everything available

print(f"Total tools loaded: {len(all_tools)}\n")
for tool in all_tools:
    spec = tool.get_tool_spec()
    desc = spec.description.split(".")[0] if spec.description else "—"
    print(f"  {tool.get_name():40s} {desc}")

---
## 3. STDIO Server

The **STDIO transport** is the standard way to connect local MCP servers to Claude Code and Claude Desktop. The server communicates via JSON-RPC 2.0 over stdin/stdout.

### How it works

```python
from orchestral.mcp import MCPServer

server = MCPServer(
    tools=tools,      # List of BaseTool instances
    name="heptapod",   # Server name shown to clients
    version="1.0.0",
)

server.run()  # Blocks, communicates via STDIO
```

### Running the server

Open a terminal and run:

```bash
# Serve all available tools
python mcp/heptapod_server_stdio.py

# Serve only specific groups
python mcp/heptapod_server_stdio.py --groups pdg,units,nda
```

You'll see the tool listing on stderr. The server then waits for JSON-RPC messages on stdin.

> **Note:** STDIO servers cannot be started from within a notebook — stdout is used for JSON-RPC, not display output. Use `server_stdio.py` from a terminal.

### Programmatic STDIO server (reference)

Here's the minimal code to create an STDIO server from scratch. This is what `server_stdio.py` does under the hood:

In [ ]:
# Reference only — do NOT run this cell (it would block the notebook)
# This shows the pattern for building a custom STDIO server.

STDIO_SERVER_CODE = '''
from orchestral.mcp import MCPServer
from tools.pdg import PDGDatabaseTool, PDGSearchTool
from tools.units import NaturalUnitsConverter

tools = [
    PDGDatabaseTool(base_directory="~/.heptapod/mcp_workspace"),
    PDGSearchTool(base_directory="~/.heptapod/mcp_workspace"),
    NaturalUnitsConverter(base_directory="~/.heptapod/mcp_workspace"),
]

server = MCPServer(tools=tools, name="heptapod-lite")
server.run()  # Blocks on STDIO
'''

print(STDIO_SERVER_CODE)

---
## 4. HTTP Server

The **HTTP transport** uses Streamable HTTP via FastMCP. This is useful for:
- Remote access (serve tools from a different machine)
- Multiple clients connecting simultaneously
- Web-based integrations

### How it works

```python
from orchestral.mcp import create_fastmcp_server

mcp = create_fastmcp_server(
    tools=tools,
    name="heptapod",
    host="127.0.0.1",
    port=8765,
    stateless_http=True,  # No session persistence between requests
)

mcp.run(transport="streamable-http")  # Blocks, serves on http://host:port/mcp
```

### Running the server

Open a terminal and run:

```bash
# Default: all tools on port 8765
python mcp/heptapod_server_http.py

# Custom configuration
python mcp/heptapod_server_http.py --host 0.0.0.0 --port 9000 --groups pdg,inspire
```

The server will be available at `http://127.0.0.1:8765/mcp`.

### Programmatic HTTP server (reference)

In [ ]:
# Reference only — do NOT run this cell (it would block the notebook)

HTTP_SERVER_CODE = '''
from orchestral.mcp import create_fastmcp_server
from tools.pdg import PDGDatabaseTool, PDGSearchTool
from tools.inspire import InspireSearchTool, InspirePaperTool

tools = [
    PDGDatabaseTool(base_directory="~/.heptapod/mcp_workspace"),
    PDGSearchTool(base_directory="~/.heptapod/mcp_workspace"),
    InspireSearchTool(base_directory="~/.heptapod/mcp_workspace"),
    InspirePaperTool(base_directory="~/.heptapod/mcp_workspace"),
]

mcp = create_fastmcp_server(
    tools=tools,
    name="heptapod-http",
    host="127.0.0.1",
    port=8765,
    stateless_http=True,
)

mcp.run(transport="streamable-http")
'''

print(HTTP_SERVER_CODE)

---
## 5. Selective Tool Export

You don't have to serve all tools at once. The group system lets you create focused servers for specific use cases.

### Example configurations

| Use Case | Groups | Description |
|----------|--------|-------------|
| Literature research | `inspire` | Paper search, citations, BibTeX |
| Quick reference | `pdg`, `units` | Particle data + unit conversions |
| Phenomenology | `pdg`, `nda`, `units` | NDA estimates with reference data |
| Model building | `feynrules_rag`, `pdg` | FeynRules generation + particle data |
| Full analysis | `analysis`, `pdg`, `units` | Event analysis pipeline |
| Everything | *(no flag)* | All available tools |

In [ ]:
# Phenomenology server: PDG + NDA + Units
pheno_tools = get_tools("pdg", "nda", "units")
print(f"Phenomenology server: {len(pheno_tools)} tools")
for t in pheno_tools:
    print(f"  - {t.get_name()}")

print()

# Literature server: INSPIRE only
lit_tools = get_tools("inspire")
print(f"Literature server: {len(lit_tools)} tools")
for t in lit_tools:
    print(f"  - {t.get_name()}")

---
## 6. Client Verification

You can verify a running server by connecting to it as an MCP client using Orchestral's `MCPClient`.

### Connecting to an STDIO server

The `MCPClient` can spawn the server as a subprocess and communicate over STDIO:

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent.parent
server_script = str(REPO_ROOT / "examples" / "mcp" / "server_stdio.py")

print(f"Server script: {server_script}")
print(f"Python:        {sys.executable}")
print()
print("To test the STDIO server as a client, run the following in a Python script:")
print(f'''
from orchestral.mcp import MCPClient

client = MCPClient(server_command=[
    "{sys.executable}",
    "{server_script}",
    "--groups", "pdg,units",
])
client.connect()

print(f"Connected! Tools: {{client.get_tool_names()}}")

# Call a tool
result = client.call_tool("PdgDatabase", {{{{
    "particle": "Higgs",
    "property": "mass",
}}}}})
print(result)

client.disconnect()
''')

### Connecting to an HTTP server

First start the HTTP server in a separate terminal:
```bash
python mcp/heptapod_server_http.py --groups pdg,units
```

Then connect from Python:

In [ ]:
print("To test the HTTP server as a client (after starting server_http.py):")
print('''
from orchestral.mcp import MCPClient

client = MCPClient(url="http://127.0.0.1:8765/mcp")
client.connect()

print(f"Connected! Tools: {client.get_tool_names()}")

# Call a tool
result = client.call_tool("PdgDatabase", {
    "particle": "top quark",
    "property": "mass",
})
print(result)

client.disconnect()
''')

---
## 7. Claude Code Integration

### STDIO Transport (recommended for local use)

Register the HEPTAPOD STDIO server with Claude Code:

```bash
# Navigate to the heptapod-dev repository
cd /path/to/heptapod-dev

# Register with all available tools
claude mcp add heptapod -- python "$(pwd)/mcp/heptapod_server_stdio.py"

# Or register with specific groups
claude mcp add heptapod -- python "$(pwd)/mcp/heptapod_server_stdio.py" --groups pdg,inspire,nda,units
```

### HTTP Transport (for remote or shared servers)

First start the HTTP server:
```bash
python mcp/heptapod_server_http.py
```

Then register with Claude Code:
```bash
claude mcp add --transport http heptapod-http http://127.0.0.1:8765/mcp
```

### Manual settings.json configuration

You can also edit `.claude/settings.json` directly:

```json
{
  "mcpServers": {
    "heptapod": {
      "command": "python",
      "args": [
        "/absolute/path/to/heptapod-dev/mcp/heptapod_server_stdio.py",
        "--groups", "pdg,inspire,nda,units"
      ]
    }
  }
}
```

Or for HTTP transport:

```json
{
  "mcpServers": {
    "heptapod-http": {
      "type": "http",
      "url": "http://127.0.0.1:8765/mcp"
    }
  }
}
```

### What Claude Code sees

Once registered, Claude Code will have access to all the HEPTAPOD tools. For example, with the `pdg` and `nda` groups, Claude can:

- Look up the Higgs boson mass: *"What is the mass of the Higgs boson?"*
- Estimate decay widths: *"Estimate the H -> bb decay width using NDA"*
- Search for particles: *"List all mesons containing a charm quark"*
- Convert units: *"Convert 125 GeV to kg"*
- Search literature: *"Find recent papers on leptoquark searches at the LHC"*

The tools appear as MCP tools with **PascalCase** names (e.g., `PdgDatabase`, `QuickNda`, `NaturalUnitsConverter`).

---
## 8. Tool Schema Reference

For reference, here are the full schemas of all lightweight tools — this is exactly what MCP clients receive when they list available tools.

In [ ]:
import json
from orchestral.mcp import MCPToolAdapter

lightweight_tools = get_tools(*LIGHTWEIGHT_GROUPS)

print(f"MCP tool schemas for {len(lightweight_tools)} lightweight tools:\n")
print("=" * 70)

for tool in lightweight_tools:
    spec = tool.get_tool_spec()
    mcp_format = MCPToolAdapter.toolspec_to_mcp(spec)
    print(f"\nTool: {mcp_format['name']}")
    print(f"Description: {mcp_format['description'][:200]}")
    print(f"Schema: {json.dumps(mcp_format['inputSchema'], indent=2)}")
    print("-" * 70)